# 02 · Swap pricing and risk

Price a USD SOFR OIS swap off the calibrated curve from notebook 01, then measure risk:

* NPV, fair rate, cashflows
* **DV01**: annuity (`fixedLegBPS`) vs parallel bump-and-reprice
* **Key-rate DV01** ladder and reconciliation to parallel DV01
* Risk profile across structures, incl. a 2s10s steepener
* NPV vs parallel shift (convexity) and a curve-twist scenario

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))   # run from notebooks/ so ql_helpers imports
import numpy as np
import QuantLib as ql
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import ql_helpers as h
import plot_utils

pio.renderers.default = "plotly_mimetype"
plot_utils.apply()
pd.options.display.float_format = "{:,.4f}".format
print("QuantLib", ql.__version__, "| valuation date", h.set_eval_date())

QuantLib 1.43 | valuation date September 25th, 2026


In [2]:
sofr = h.SofrCurve()

def make_swap(tenor="5Y", fixed=0.0350, notional=100e6, payer=True, fwd="0D"):
    return ql.MakeOIS(ql.Period(tenor), sofr.index, fixed, ql.Period(fwd),
                      nominal=notional, receiveFixed=not payer,
                      discountingTermStructure=sofr.handle, paymentLag=2)

swap = make_swap()
print("type      :", "Payer" if swap.type() == ql.Swap.Payer else "Receiver")
print("start/end :", swap.startDate().ISO(), "->", swap.maturityDate().ISO())
print("fixed rate: %.4f%% | fair rate: %.4f%%" % (swap.fixedRate() * 100, swap.fairRate() * 100))
print("NPV       :", f"{swap.NPV():,.0f}")
print("fixed leg NPV / SOFR leg NPV:", f"{swap.fixedLegNPV():,.0f}", "/", f"{swap.overnightLegNPV():,.0f}")

type      : Payer
start/end : 2026-09-29 -> 2031-09-29
fixed rate: 3.5000% | fair rate: 3.7300%
NPV       : 1,045,898
fixed leg NPV / SOFR leg NPV: -15,915,844 / 16,961,743


## Cashflows

Fixed leg shown explicitly; the SOFR leg is compounded overnight, so its coupon amounts are curve-dependent.

In [3]:
def leg_table(leg):
    return pd.DataFrame([{"pay_date": cf.date().ISO(), "amount": cf.amount(),
                          "df": sofr.curve.discount(cf.date()),
                          "pv": cf.amount() * sofr.curve.discount(cf.date())} for cf in leg])

fixed, flt = leg_table(swap.fixedLeg()), leg_table(swap.overnightLeg())
display(fixed.style.format({"amount": "{:,.0f}", "df": "{:.6f}", "pv": "{:,.0f}"}).set_caption("Fixed leg (paid)"))
display(flt.style.format({"amount": "{:,.0f}", "df": "{:.6f}", "pv": "{:,.0f}"}).set_caption("SOFR leg (received)"))
print("PV(float) - PV(fixed) =", f"{flt.pv.sum() - fixed.pv.sum():,.0f}", "| swap NPV =", f"{swap.NPV():,.0f}")

,pay_date,amount,df,pv
0,2027-10-01,"3,548,611",0.963036,"3,417,439"
1,2028-10-03,"3,558,333",0.929802,"3,308,545"
2,2029-10-02,"3,538,889",0.896771,"3,173,574"
3,2030-10-02,"3,568,056",0.863230,"3,080,052"
4,2031-10-01,"3,538,889",0.829705,"2,936,234"


,pay_date,amount,df,pv
0,2027-10-01,"3,771,667",0.963036,"3,632,249"
1,2028-10-03,"3,554,653",0.929802,"3,305,123"
2,2029-10-02,"3,681,651",0.896771,"3,301,599"
3,2030-10-02,"3,905,190",0.863230,"3,371,076"
4,2031-10-01,"4,039,621",0.829705,"3,351,695"


PV(float) - PV(fixed) = 1,045,898 | swap NPV = 1,045,898


## DV01: annuity vs bump-and-reprice

`fixedLegBPS` is the fixed-leg PV change per 1bp of fixed rate (the annuity). Bump-and-reprice shifts the curve, which also moves the floating leg. For a payer swap, DV01 (per bp *up* in rates) is positive.

In [4]:
def dv01_parallel(s, bp=1.0):
    """Central-difference DV01: NPV change per +1bp parallel shift of all OIS quotes."""
    sofr.bump(None, +bp / 2); up = s.NPV()
    sofr.reset(); sofr.bump(None, -bp / 2); dn = s.NPV()
    sofr.reset()
    return (up - dn) / bp

annuity = abs(swap.fixedLegBPS())
parallel = dv01_parallel(swap)
print("Annuity (|fixedLegBPS|, per bp)  :", f"{annuity:,.0f}")
print("Parallel bump DV01 (payer, per bp):", f"{parallel:,.0f}")

Annuity (|fixedLegBPS|, per bp)  : 45,474
Parallel bump DV01 (payer, per bp): 45,172


## Key-rate DV01 ladder

Bump each OIS input quote by 1bp and reprice. The sum of key-rate DV01s should reconcile to the parallel DV01.

In [5]:
def key_rate_dv01(s, bp=1.0):
    base = s.NPV()
    out = {}
    for tenor in sofr.quotes:
        sofr.bump(tenor, bp)
        out[tenor] = (s.NPV() - base) / bp
        sofr.reset()
    return pd.Series(out, name="DV01_usd_per_bp")

krd = key_rate_dv01(swap)
display(krd.to_frame().style.format("{:,.0f}"))
print("sum of KRDs:", f"{krd.sum():,.0f}", "| parallel:", f"{parallel:,.0f}", "| diff:", f"{krd.sum() - parallel:,.1f}")

fig = go.Figure(go.Bar(x=krd.index, y=krd.values))
fig.update_layout(title="Key-rate DV01 · 5Y payer @ 3.50%, $100mm", yaxis_title="USD / bp", height=380)
fig.show()

,DV01_usd_per_bp
1W,-1
1M,0
3M,-0
6M,0
1Y,-19
2Y,-39
3Y,-59
4Y,-81
5Y,"45,367"
7Y,0


sum of KRDs: 45,168 | parallel: 45,172 | diff: -4.4


## Risk across structures

KRD profiles for 2Y / 5Y / 10Y payers and a DV01-neutral 2s10s steepener (pay 10Y, receive 2Y).

In [6]:
def par_swap(tenor, payer=True, notional=100e6):
    return make_swap(tenor, fixed=sofr.quote_levels[tenor], notional=notional, payer=payer)

s10 = par_swap("10Y", payer=True)
ratio = abs(dv01_parallel(s10)) / abs(dv01_parallel(par_swap("2Y", payer=False)))
s2 = par_swap("2Y", payer=False, notional=100e6 * ratio)

profiles = pd.DataFrame({
    "pay 2Y": key_rate_dv01(par_swap("2Y")),
    "pay 5Y": key_rate_dv01(par_swap("5Y")),
    "pay 10Y": key_rate_dv01(s10),
    "2s10s steepener": key_rate_dv01(s10) + key_rate_dv01(s2),
})
display(profiles.style.format("{:,.0f}"))
print("2s10s: receive 2Y notional = $%.1fmm | parallel DV01 = %s" % (100 * ratio, f"{dv01_parallel(s10) + dv01_parallel(s2):,.1f}"))

fig = go.Figure([go.Bar(x=profiles.index, y=profiles[c], name=c) for c in profiles.columns])
fig.update_layout(barmode="group", title="Key-rate DV01 by structure", yaxis_title="USD / bp", height=420)
fig.show()

,pay 2Y,pay 5Y,pay 10Y,2s10s steepener
1W,0,-0,-0,-0
1M,0,-0,-0,-0
3M,-0,0,-0,0
6M,-0,-0,-0,-0
1Y,-0,0,-0,-0
2Y,"19,215",0,-0,"-82,600"
3Y,-0,0,-0,-0
4Y,-0,0,-0,-0
5Y,-0,"45,469",-0,-0
7Y,-0,-0,-0,-0


2s10s: receive 2Y notional = $429.9mm | parallel DV01 = -0.0


## Scenarios: parallel shifts (convexity) and a twist

In [7]:
base_npv = swap.NPV()
rows = []
for sh in range(-100, 101, 25):
    sofr.bump(None, sh)
    rows.append({"shift_bp": sh, "NPV": swap.NPV()})
    sofr.reset()
sc = pd.DataFrame(rows).set_index("shift_bp")
sc["linear (DV01)"] = base_npv + parallel * sc.index
sc["convexity_pnl"] = sc["NPV"] - sc["linear (DV01)"]
display(sc.style.format("{:,.0f}"))

twist = {"1Y": -25, "2Y": -25, "3Y": -15, "5Y": 0, "7Y": 10, "10Y": 25, "15Y": 25, "20Y": 25, "30Y": 25}
for t, bp in twist.items():
    sofr.bump(t, bp)
print("Steepener twist (short end -25bp, long end +25bp): 5Y payer P&L =", f"{swap.NPV() - base_npv:,.0f}")
sofr.reset()

fig = go.Figure()
fig.add_scatter(x=sc.index, y=sc["NPV"], name="full reval")
fig.add_scatter(x=sc.index, y=sc["linear (DV01)"], name="DV01 approx", line=dict(dash="dash"))
fig.update_layout(title="5Y payer NPV vs parallel shift", xaxis_title="bp", yaxis_title="USD", height=400)
fig.show()

,NPV,linear (DV01),convexity_pnl
shift_bp,,,
-100,"-3,604,708","-3,471,348","-133,360"
-75,"-2,416,622","-2,342,036","-74,586"
-50,"-1,245,685","-1,212,725","-32,961"
-25,"-91,607","-83,413","-8,193"
0,"1,045,898","1,045,898",0
25,"2,167,108","2,175,210","-8,101"
50,"3,272,297","3,304,521","-32,224"
75,"4,361,734","4,433,833","-72,099"
100,"5,435,680","5,563,144","-127,464"


Steepener twist (short end -25bp, long end +25bp): 5Y payer P&L = 2,342
